# EcoMove Mobility – Data Visualisation Process
COM7021 [5442]. Step-by-step run of the analysis: load, inspect, clean, summarise, visualise.

## Step 1 – Load the raw dataset

In [ ]:
import pandas as pd
raw = pd.read_excel('data/EcoMoveMobility.xlsx')
print('Shape:', raw.shape)
raw.head()

## Step 2 – Inspect data types and structure

In [ ]:
raw.info()

## Step 3 – Data quality checks

In [ ]:
print('Missing values per column:')
print(raw.isna().sum().to_string())
print('\nDuplicate rows:', raw.duplicated().sum())
print('Duplicate Record_IDs:', raw['Record_ID'].duplicated().sum())
print('\nDate column dtype:', raw['Date'].dtype, '| sample values:', raw['Date'].head(3).tolist())

## Step 4 – Clean the data and derive measures
Excel serial dates are converted to datetimes; Month, Profit and Cost-recovery are derived.

In [ ]:
from ecomove_analysis import load_data, data_quality_report
df = load_data()
df[['Date', 'Month', 'Ticket_Revenue_EUR', 'Operating_Cost_EUR', 'Profit_EUR', 'Cost_Recovery', 'Event_Day']].head()

In [ ]:
data_quality_report(df)

## Step 5 – Summary tables by transport mode and country

In [ ]:
agg = dict(Passengers=('Passengers','sum'), Revenue=('Ticket_Revenue_EUR','sum'), Cost=('Operating_Cost_EUR','sum'),
           Avg_delay=('Average_Delay_Minutes','mean'), On_time=('On_Time_Percentage','mean'), Rating=('Customer_Rating','mean'))
mode = df.groupby('Transport_Mode').agg(**agg)
mode['Cost_recovery'] = mode.Revenue / mode.Cost
mode.sort_values('Passengers', ascending=False).round(2)

In [ ]:
country = df.groupby('Country').agg(**agg, Cities=('City','nunique'))
country['Cost_recovery'] = country.Revenue / country.Cost
country.sort_values('Passengers', ascending=False).round(2)

## Step 6 – Create the eight visualisations

In [ ]:
import ecomove_analysis as ea
for f in (ea.fig1_time_series, ea.fig2_city_demand, ea.fig3_correlation, ea.fig4_delay_distribution,
          ea.fig5_cost_recovery, ea.fig6_accessibility_heatmap, ea.fig7_delay_vs_rating, ea.fig8_event_days):
    f(df)

## Step 7 – Launch the interactive dashboard
Run `python ecomove_dashboard.py` and open http://127.0.0.1:8050